# EDA: Trader Performance vs Market Sentiment
This notebook analyzes trader performance segmented by the crypto Fear & Greed index.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load datasets
fear_greed_df = pd.read_csv('Datasets/fear_greed_index.csv')
trades_df = pd.read_csv('Datasets/historical_data.csv')


In [2]:
# Preprocess Fear & Greed Index
fear_greed_df['date'] = pd.to_datetime(fear_greed_df['date'])

# Preprocess Trade Data
# Timestamp IST looks like '02-12-2024 22:50'
trades_df['Timestamp IST'] = pd.to_datetime(trades_df['Timestamp IST'], format='mixed', dayfirst=True)
trades_df['date'] = trades_df['Timestamp IST'].dt.normalize()

# Merge Datasets
merged_df = pd.merge(trades_df, fear_greed_df, on='date', how='left')

# Inspect shapes and sample
print(f'Fear Greed shape: {fear_greed_df.shape}')
print(f'Trades shape: {trades_df.shape}')
print(f'Merged shape: {merged_df.shape}')
merged_df.head()


Fear Greed shape: (2644, 4)
Trades shape: (211224, 17)
Merged shape: (211224, 20)


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,date,timestamp,value,classification
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,2024-12-02 22:50:00,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,2024-12-02 22:50:00,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,2024-12-02 22:50:00,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,2024-12-02 22:50:00,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,2024-12-02 22:50:00,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12,2024-12-02,1.733117e+09,80.0,Extreme Greed


In [3]:
# Basic metrics for PnL
# Clean PnL
merged_df['Closed PnL'] = pd.to_numeric(merged_df['Closed PnL'], errors='coerce').fillna(0)

# Create a simple categorization for standardizing fear vs greed
def map_sentiment(val):
    if val < 25:
        return 'Extreme Fear'
    elif val < 45:
        return 'Fear'
    elif val < 55:
        return 'Neutral'
    elif val < 75:
        return 'Greed'
    else:
        return 'Extreme Greed'

merged_df['Sentiment_Group'] = merged_df['value'].apply(map_sentiment)


In [4]:
# Summary Statistics by Sentiment
summary = merged_df.groupby('Sentiment_Group').agg(
    Trade_Count=('Trade ID', 'count'),
    Avg_PnL=('Closed PnL', 'mean'),
    Total_PnL=('Closed PnL', 'sum'),
    Win_Rate=('Closed PnL', lambda x: (x > 0).mean() * 100),
    Avg_Trade_Size=('Size USD', 'mean')
).reset_index()

summary


,Sentiment_Group,Trade_Count,Avg_PnL,Total_PnL,Win_Rate,Avg_Trade_Size
0,Extreme Fear,21400,34.537862,7.391102e+05,37.060748,5349.731843
1,Extreme Greed,39998,68.944530,2.757643e+06,46.502325,3114.001536
2,Fear,61837,54.290400,3.357155e+06,42.076750,7816.109931
3,Greed,50303,42.743559,2.150129e+06,38.482794,5736.884375
4,Neutral,37686,34.307718,1.292921e+06,39.699093,4782.732661
